# Milvus

In [4]:
from pymilvus import MilvusClient

host = "localhost"
port = "19530"

milvus_client = MilvusClient(host=host, port=port)

In [5]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(
    name="id", dtype=DataType.INT64, is_primary=True, description="Primary id"
)
text = FieldSchema(
    name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text"
)
embedding_text = FieldSchema(
    "embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=VECTOR_LENGTH,
    description="Embedded text",
)

fields = [id_field, text, embedding_text]

schema = CollectionSchema(
    fields=fields,
    auto_id=True,
    enable_dynamic_field=True,
    description="RAG Texts collection",
)

In [6]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(collection_name=COLLECTION_NAME, schema=schema)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64},  # lower values for speed
)

milvus_client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 470098147540993300, 'consistency_level': 2, 'consistency_level_name': 'Bounded', 'properties': {'timezone': 'UTC'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'enable_namespace': False, 'schema_version': 0, 'created_timestamp': 470098162702876690, 'update_timestamp': 470098162702876690}


In [7]:
# define data source and destination
## the document origin destination from which document will be downloaded
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [8]:
# download data
import os
import requests


def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)


download_pdf_data(pdf_url, file_name)

In [12]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)


extract_pdf_text(file_name, file_json)

In [13]:
# vectorize data

import torch
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append(
            {"page_num": page_num, "embedding": embeddings[page_num].tolist()}
        )

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)


model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

/home/maja/Studia/Magisterskie/Semestr-1/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2987.77it/s]


In [14]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with (
        open(os.path.join(data_dir, file_json), "r") as t_f,
        open(os.path.join(data_dir, embeddings_json), "r") as e_f,
    ):
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data = list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))

        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text": text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [32]:
# search


def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings",
        data=[embedded_query],
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"],
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

In [33]:
result

data: [[{'id': 470098147540995426, 'distance': 29.12516975402832, 'entity': {'text': 'Historia powstania\nsztucznej inteligencji\n7\nW języku potocznym „sztuczny" oznacza to, co\njest \nwytworem \nmającym \nnaśladować \ncoś\nnaturalnego. W takim znaczeniu używamy\nterminu ,,sztuczny\'\', gdy mówimy o sztucznym\nlodowisku lub oku. Sztuczna inteligencja byłaby\nczymś (programem, maszyną) symulującym\ninteligencję naturalną, ludzką.\nSztuczna inteligencja (AI) to obszar informatyki,\nktóry skupia się na tworzeniu programów\nkomputerowych zdolnych do wykonywania\nzadań, które wymagają ludzkiej inteligencji. \nTe zadania obejmują rozpoznawanie wzorców,\nrozumienie języka naturalnego, podejmowanie\ndecyzji, uczenie się, planowanie i wiele innych.\nGłównym celem AI jest stworzenie systemów,\nktóre są zdolne do myślenia i podejmowania\ndecyzji na sposób przypominający ludzki.\nHistoria sztucznej inteligencji sięga lat 50. \nXX wieku, kiedy to powstały pierwsze koncepcje\ni modele tego, co mogł

## RAG

In [27]:
from dotenv import load_dotenv

load_dotenv()

True

In [28]:
!ls

alchemy_db.ipynb  data	milvus_db  milvus.ipynb  vectorscale_db


In [42]:
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-3.5-flash-lite"  # changed from 2.5-flash


def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [49]:
def build_prompt(context: str, query: str) -> str:
    context = context[0][0]["entity"]["text"]
    return f"Kontekst: {context}\nPytanie: {query}"


def rag(model, query: str) -> str:
    context = search(model, query)
    prompt = build_prompt(context, query)
    response = generate_response(prompt)
    return response

In [50]:
answer = rag(model, "Czym jest sztuczna inteligencja?")
print(answer)

Na podstawie dostarczonego tekstu, sztuczna inteligencja (AI) to:

1. **W ujęciu ogólnym:** Coś (program lub maszyna) symulujące inteligencję naturalną, ludzką. Jest to system, który ma realizować niektóre funkcje umysłu (czasami w sposób przewyższający funkcje naturalne, np. bez pomyłek w liczeniu czy defektów pamięci).
2. **W ujęciu informatycznym:** Obszar informatyki skupiający się na tworzeniu programów komputerowych zdolnych do wykonywania zadań wymagających ludzkiej inteligencji (takich jak rozpoznawanie wzorców, rozumienie języka naturalnego, podejmowanie decyzji, uczenie się, planowanie itp.).
3. **W ujęciu celu:** Systemy zdolne do myślenia i podejmowania decyzji w sposób przypominający ludzki.

*Uwaga: Współczesne systemy sztucznej inteligencji są jednak inteligentne tylko w ograniczonym obszarze.*


In [56]:
answer = rag(
    model,
    "Czym jest sztuczna inteligencja? Odpowiedz w mniej niż 50 słowach, nie używając litery `e`. Upewnij się, że Twoja wypowiedź nie zawiera wystąpień litery `e`.",
)
print(answer)

Sztuczna inteligencja to dziedzina nauki i techniki. Obejmuje programy oraz systemy zdolne do wykonywania trudnych zadań, nauki, analizy danych, a takze symulacji ludzkiego myślenia. Pomaga w rozwiązywaniu problemów, usprawniając codzienne działania ludzi.


In [57]:
len(answer.split())

32

In [59]:
"e" in answer  # constraint not satisfied

True

In [43]:
answer = rag(model, "Czym jest Tree of Thoughts?")
print(answer)

Na podstawie dostarczonego tekstu **nie ma żadnych informacji na temat "Tree of Thoughts"**. 

Tekst omawia wyłącznie następujące rodzaje sztucznej inteligencji:
1. Sztuczna inteligencja słaba (wąska)
2. Sztuczna inteligencja silna
3. Sztuczna inteligencja umysłowa
4. Sztuczna inteligencja zwiększająca (Augmented Intelligence)
5. Sztuczna inteligencja bezpieczna (Safe AI)
6. Sztuczna inteligencja odwrócona (Inverse AI)
7. Sztuczna inteligencja reprezentacyjna (Representational AI)
8. Sztuczna inteligencja predykcyjna (Predictive AI)
